# Tiny GPT — pretraining on a free Colab GPU

This notebook pretrains the **from-scratch** GPT in this repo (same hand-written
attention/LayerNorm/etc.) on a real text corpus, at ~60M parameters, using a
free Colab **T4** GPU.

First: **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

Expect the model to go from gibberish to coherent little stories over a few
hours. Checkpoints are saved every eval, so you can stop and resume.

In [ ]:
# 1. Confirm we have a GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi -L

In [ ]:
# 2. Get the code and install deps.
# (If the repo is private, either make it public or add a token to the URL.)
!git clone -b claude/gpt-model-pytorch-scratch-ckb4u7 https://github.com/mitchellmcneillharrison-png/Smallchatbot.git
%cd Smallchatbot
!pip install -q -r requirements.txt

In [ ]:
# 3. Download a pretraining corpus. TinyStories is purpose-built so small
#    models learn coherent English. 200 MB is plenty for a first run.
!python data/get_pretrain_data.py --dataset tinystories --max_mb 200

In [ ]:
# 4. Pretrain a ~57M-parameter GPT (12 layers would give ~85M).
#    fp16 is what the T4 supports; grad_accum gives a larger effective batch.
#    If you hit out-of-memory, lower --batch_size (e.g. 16) or --block_size (e.g. 192).
!python train.py \
  --data_path data/pretrain.txt \
  --block_size 256 --n_layer 8 --n_head 12 --n_embd 768 \
  --batch_size 32 --grad_accum 4 --amp fp16 \
  --lr 3e-4 --min_lr 3e-5 --warmup_steps 300 --max_steps 20000 \
  --eval_interval 1000 --eval_iters 50 \
  --sample_prompt "Once upon a time"

In [ ]:
# 5. Generate from the trained checkpoint.
!python generate.py --checkpoint checkpoints/ckpt.pt \
  --prompt "Once upon a time" --max_new_tokens 500 --temperature 0.8 --top_k 50

## Notes

- **Time / free-tier limits.** A T4 trains a ~57M char model at a few thousand
  tokens/sec; coherent TinyStories text emerges after a couple of hours. Colab
  free sessions disconnect when idle (~90 min) and cap total runtime (~12 h),
  so keep the tab active and **resume** if it stops.
- **Resume:** re-run the training cell with `--resume checkpoints/ckpt.pt` added.
- **Save your model:** download `checkpoints/ckpt.pt`, or mount Drive
  (`from google.colab import drive; drive.mount('/content/drive')`) and point
  `--out_dir` there so checkpoints survive a disconnect.
- **Bigger:** `--n_layer 12` → ~85M; also try `--block_size 384`. Watch GPU
  memory (`!nvidia-smi`).
- **Faster coherence:** a smaller model (`--n_embd 512 --n_layer 6`, ~19M)
  reaches readable text sooner if you're impatient.